# Masiez — Train a PPO Target-Following Policy (Kaggle) → ONNX for the Browser

Train a small PPO policy that **closes in on a detected target and holds station on it**, using only what the browser pipeline can provide:

- **Observation** `float32[1, 9]` (the contract with `game/src/ai/PpoFollowPolicy.ts`):
  `[dx, dy, w, h, vf, vr, vu, rangeEst, altDelta]`
  - `dx, dy` — target box center offset from image center, each -0.5..0.5 (right / down positive)
  - `w, h` — target box width/height as a fraction of the frame, 0..1
  - `vf, vr, vu` — drone velocity in the **camera frame** (forward, right, up) divided by 20 m/s
  - `rangeEst` — ground range to the target **estimated from the apparent box height**, divided by 100 m, clipped to 0..2. It is the estimate on purpose: the deployed policy must see the same noisy signal it trained on.
  - `altDelta` — drone altitude above the target center, divided by 25 m, clipped to -1..1
- **Action** `float32[1, 3]` in -1..1 — velocity commands `[forward, strafe, climb]`, which the browser maps onto the drone's pitch/roll/throttle axes.

Heading is auto-aligned to the target by a yaw PD assist (the same assist the browser applies), so the policy only outputs velocity commands. The reward closes the distance, then **loiter inside a ring** instead of flying through the target, and punishes sinking into the ground.

Run on Kaggle (the MLP also trains on CPU), then download `ppo-follow.onnx` into `game/public/models/`. An older 7-float export still loads — the browser zero-pads the range and altitude features.


In [ ]:
%pip install -q torch numpy matplotlib onnx onnxruntime onnxscript

## 1. Flight environment

A point-mass drone chases a wandering ground target, then holds station on it. The observation is built with the same pinhole optics as the game FPV camera (84 degrees vertical, horizontal derived from a 16:9 frame), and the range feature is inverted from the apparent box height exactly as the browser does it. The airframe flies inside a comfort band between `MIN_ALT` and `CEIL_ALT`: sinking costs the episode, climbing out of the band costs reward, because both push the detector box and the derived range into regimes the policy cannot read.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, os, shutil

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else './ppo_work'
os.makedirs(WORK, exist_ok=True)

MAX_SPEED = 20.0
# Optics mirror the game FPV camera: 84 degrees vertical, derived horizontal.
V_FOV = math.radians(84.0)
ASPECT = 16 / 9
H_FOV = 2 * math.atan(math.tan(V_FOV / 2) * ASPECT)
TARGET_W = 2.5            # metres; the browser assumes the same box
TARGET_H = 2.0
RANGE_NORM = 100.0        # normaliser for the estimated-range feature
ALT_NORM = 30.0           # normaliser for the altitude-delta feature
LOITER_R = 12.0           # capture ring radius, metres (a 20 m/s orbit needs room)
MIN_ALT = 5.0             # below this the airframe is in danger
WORK_ALT = 20.0           # comfort band top; the ceiling keeps features in range
CEIL_ALT = 30.0
CAPTURE_HITS = 12         # steps held inside the ring to call it a capture
EPISODE_STEPS = 400
OBS_DIM = 9

class FollowEnv:
    """Point-mass chase. The only sensors are the detector box and own velocity,
    so every feature has to be derivable from what the camera really sees."""

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.pos = np.array([0.0, float(self.rng.uniform(MIN_ALT + 1.0, WORK_ALT)), 0.0])
        self.vel = np.zeros(3)
        self.yaw = self.rng.uniform(-math.pi, math.pi)
        ang = self.rng.uniform(-math.pi, math.pi)
        d = self.rng.uniform(15.0, 60.0)
        self.target = np.array([d * math.cos(ang), 0.0, d * math.sin(ang)])
        self.tvel = self.rng.uniform(-2.5, 2.5, 3) * np.array([1, 0, 1])
        self.dist = float(np.linalg.norm(self.target - self.pos))
        self.loiter = 0
        self.steps = 0
        self.crashed = False
        return self._obs()

    def _bearing(self):
        rel = self.target - self.pos
        return math.atan2(rel[0], rel[2])

    def _obs(self):
        rel = self.target - self.pos
        c, s = math.cos(self.yaw), math.sin(self.yaw)
        f = rel[0] * s + rel[2] * c          # forward (matches camera heading)
        r = rel[0] * c - rel[2] * s          # right
        u = -rel[1]                          # target below the drone reads as "down"
        d = max(float(np.linalg.norm(rel)), 1e-3)

        if f > 1.0:
            dx = float(np.clip((r / f) / (2 * math.tan(H_FOV / 2)), -0.5, 0.5))
            dy = float(np.clip((u / f) / (2 * math.tan(V_FOV / 2)), -0.5, 0.5))
            w = float(np.clip(TARGET_W / (2 * f * math.tan(H_FOV / 2)), 0.005, 1.0))
            h = float(np.clip(TARGET_H / (2 * f * math.tan(V_FOV / 2)), 0.005, 1.0))
            # Range comes from the apparent box height, which is exactly what the
            # browser can compute, so training and deployment feed one quantity.
            range_est = TARGET_H / (2 * h * math.tan(V_FOV / 2))
        else:                                 # behind the camera: no usable box
            dx = 0.5 * (1.0 if r >= 0 else -1.0)
            dy, w, h = 0.0, 0.005, 0.005
            range_est = d

        alt_delta = self.pos[1] - (self.target[1] + TARGET_H / 2)
        vf = self.vel[0] * s + self.vel[2] * c
        vr = self.vel[0] * c - self.vel[2] * s
        vu = self.vel[1]
        return np.array([
            dx, dy, w, h,
            vf / MAX_SPEED, vr / MAX_SPEED, vu / MAX_SPEED,
            float(np.clip(range_est / RANGE_NORM, 0.0, 2.0)),
            float(np.clip(alt_delta / ALT_NORM, -1.0, 1.0)),
        ], np.float32)

    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        prev = self.dist
        self.steps += 1

        # yaw PD assist keeps the camera on target (the browser does the same)
        err = math.atan2(math.sin(self._bearing() - self.yaw), math.cos(self._bearing() - self.yaw))
        self.yaw += float(np.clip(err * 3.0, -2.0, 2.0)) / 30.0

        c, s = math.cos(self.yaw), math.sin(self.yaw)
        want = np.array([action[0] * s + action[1] * c, action[2], action[0] * c - action[1] * s]) * MAX_SPEED
        self.vel += (want - self.vel) * min(1.0, 6.0 / 30.0)
        self.pos += self.vel / 30.0

        # A hard ceiling keeps the box-height range estimate inside the range the
        # policy was trained on; without it the drone can climb into saturated
        # observations and fly blind.
        if self.pos[1] > CEIL_ALT:
            self.pos[1] = CEIL_ALT

        self.target += self.tvel / 30.0
        self.target[1] = 0.0
        if self.rng.random() < 0.02:
            self.tvel = self.rng.uniform(-2.5, 2.5, 3) * np.array([1, 0, 1])
        self.dist = float(np.linalg.norm(self.target - self.pos))

        # Closing is shaped smoothly, not just as a step difference: a drone that
        # stalls at a standoff distance gets no gradient from (prev - dist) alone.
        reward = (prev - self.dist) * 2.0 - 0.005 + 0.6 * max(0.0, 1.0 - self.dist / 30.0)

        # Anti-suicide: the ground is at y = 0 and touching it ends the episode.
        if self.pos[1] <= 0.5:
            self.pos[1] = 0.5
            self.crashed = True
            reward -= 120.0
        elif self.pos[1] < MIN_ALT:
            reward -= 0.5 * (MIN_ALT - self.pos[1])        # climb out of the danger band
        elif self.pos[1] > WORK_ALT:
            reward -= 0.35 * (self.pos[1] - WORK_ALT)      # come down into the work band

        # Capture is a loiter, not a collision: hold the ring, do not fly through it.
        if self.dist <= LOITER_R:
            self.loiter += 1
            reward += 1.5
            if self.dist < LOITER_R * 0.4:
                reward -= 2.0                              # too close risks a strike
        else:
            self.loiter = 0

        caught = self.loiter >= CAPTURE_HITS
        done = caught or self.crashed or self.steps >= EPISODE_STEPS
        if caught:
            reward += 150.0
        return self._obs(), reward, done, {
            'caught': caught, 'crashed': self.crashed, 'dist': self.dist, 'alt': float(self.pos[1]),
        }


## 2. PPO (actor-critic MLP over 9 observations, GAE, clipped objective)

In [ ]:
torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def mlp(sizes):
    layers = []
    for i in range(len(sizes) - 1):
        layers += [nn.Linear(sizes[i], sizes[i + 1])]
        if i < len(sizes) - 2:
            layers += [nn.Tanh()]
    return nn.Sequential(*layers)

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = mlp([OBS_DIM, 64, 64])
        self.mu = nn.Linear(64, 3)
        self.logstd = nn.Parameter(torch.zeros(3))
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        z = self.backbone(x)
        return self.mu(z), self.logstd.expand_as(self.mu(z)), self.v(z)

pi = ActorCritic().to(DEV)
opt = torch.optim.Adam(pi.parameters(), lr=3e-4)
GAMMA, LAMBDA, CLIP, ENT_W, VCLIP = 0.99, 0.95, 0.2, 0.01, 10.0

def run_updates(n_updates=250, steps_per_env=150, n_envs=8):
    envs = [FollowEnv(seed=i) for i in range(n_envs)]
    obs = np.stack([e.reset() for e in envs])
    total_caught = 0
    total_crashed = 0
    for upd in range(n_updates):
        buf_o, buf_a, buf_lp, buf_r, buf_d, buf_v = [], [], [], [], [], []
        for _ in range(steps_per_env):
            t = torch.tensor(obs, dtype=torch.float32, device=DEV)
            with torch.no_grad():
                mu, std, v = pi(t)
                a = torch.distributions.Normal(mu, std.exp()).sample()
            a_np = a.cpu().numpy()
            new_obs, rewards, dones = [], [], []
            for i, e in enumerate(envs):
                o2, r, d, info = e.step(a_np[i])
                if d:
                    total_caught += int(info['caught'])
                    total_crashed += int(info['crashed'])
                    o2 = e.reset()
                new_obs.append(o2); rewards.append(r); dones.append(float(d))
            buf_o.append(obs); buf_a.append(a_np)
            buf_lp.append(torch.distributions.Normal(mu, std.exp()).log_prob(a).sum(-1).detach().cpu().numpy())
            buf_r.append(np.array(rewards)); buf_d.append(np.array(dones)); buf_v.append(v.detach().squeeze(-1).cpu().numpy())   # (envs,) or GAE broadcasts to (envs, envs)
            obs = np.stack(new_obs)

        # GAE (time-major, envs inside each step)
        R = np.zeros(n_envs); advs = []; rets = []
        r_arr = np.stack(buf_r); v_arr = np.stack(buf_v); d_arr = np.stack(buf_d)
        for t in reversed(range(steps_per_env)):
            delta = r_arr[t] + GAMMA * R * (1 - d_arr[t]) - v_arr[t]
            R = delta + GAMMA * LAMBDA * (1 - d_arr[t]) * R
            advs.insert(0, R); rets.insert(0, R + v_arr[t])
        advs = np.concatenate(advs)
        advs = (advs - advs.mean()) / (advs.std() + 1e-8)
        all_o = np.concatenate(buf_o)
        all_a = np.concatenate(buf_a)
        all_lp = np.concatenate(buf_lp)
        all_ret = np.concatenate(rets)

        for _ in range(10):
            t_o = torch.tensor(all_o, device=DEV); t_a = torch.tensor(all_a, device=DEV)
            t_lp = torch.tensor(all_lp, device=DEV); t_ad = torch.tensor(advs, device=DEV)
            t_ret = torch.tensor(all_ret, device=DEV)
            mu, std, v = pi(t_o)
            dist = torch.distributions.Normal(mu, std.exp())
            lp = dist.log_prob(t_a).sum(-1)
            ratio = (lp - t_lp).exp()
            surr1 = ratio * t_ad
            surr2 = torch.clamp(ratio, 1 - CLIP, 1 + CLIP) * t_ad
            vf_loss = F.mse_loss(v.squeeze(-1).clamp(-100, 100), t_ret.clamp(-100, 100))
            ent = dist.entropy().sum(-1).mean()
            loss = -torch.min(surr1, surr2).mean() + 0.5 * vf_loss - ENT_W * ent
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(pi.parameters(), 0.5)
            opt.step()

        if upd % 20 == 19:
            d_now = np.mean([e.dist for e in envs])
            print(f'update {upd+1:3d} | mean dist {d_now:6.1f} m | loitered {total_caught} | ground strikes {total_crashed}')
    return envs

envs = run_updates()

## 3. Rollout check

In [ ]:
def greedy_rollout(seed=99, max_steps=EPISODE_STEPS):
    e = FollowEnv(seed=seed)
    o = e.reset()
    pi.eval()
    with torch.no_grad():
        for t in range(max_steps):
            mu, _, _ = pi(torch.tensor(o[None], dtype=torch.float32, device=DEV))
            o, r, done, info = e.step(mu.cpu().numpy()[0])
            if done:
                return info, t + 1
    return {'caught': False, 'crashed': False, 'dist': e.dist, 'alt': e.pos[1]}, max_steps

for seed in range(8):
    info, steps = greedy_rollout(seed)
    outcome = 'LOITER' if info['caught'] else ('GROUND STRIKE' if info['crashed'] else 'timeout')
    print(f"seed {seed}: {outcome:12s} in {steps:3d} steps  (dist {info['dist']:5.1f} m, alt {info['alt']:4.1f} m)")


## 4. Export actor to ONNX

Export **only the action mean** (deterministic policy) with the fixed browser interface: input `obs` float32[1,9] → output `action` float32[1,3] with a `tanh` squash so commands always land in -1..1.

In [ ]:
class DeployablePolicy(nn.Module):
    def __init__(self, actor):
        super().__init__()
        self.net = nn.Sequential(actor.backbone, actor.mu, nn.Tanh())
    def forward(self, x):
        return self.net(x)

deploy = DeployablePolicy(pi).to(DEV).eval()
onnx_file = os.path.join(WORK, 'ppo-follow.onnx')
dummy = torch.zeros(1, OBS_DIM, device=DEV)
torch.onnx.export(deploy, dummy, onnx_file,
                  input_names=['obs'], output_names=['action'],
                  opset_version=14, dynamic_axes=None)
print('Exported:', onnx_file)


## 5. Verify the exported policy

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(onnx_file, providers=['CPUExecutionProvider'])
print('input:', sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print('output:', sess.get_outputs()[0].name, sess.get_outputs()[0].shape)
assert sess.get_inputs()[0].shape[1] == OBS_DIM, 'observation width drift'
for seed in range(5):
    e = FollowEnv(seed=seed)
    o = e.reset()
    with torch.no_grad():
        mu = pi(torch.tensor(o[None], dtype=torch.float32, device=DEV))[0]
    onnx_a = sess.run(None, {'obs': o[None].astype(np.float32)})[0]
    torch_a = np.tanh(mu.cpu().numpy())
    assert np.allclose(onnx_a, torch_a, atol=1e-4), 'mismatch!'
print('ONNX matches PyTorch actor — OK')


## 6. Ship it to the browser

Download `ppo-follow.onnx` from the output panel and place it in `game/public/models/ppo-follow.onnx` (git-ignored). In the game's User Menu set **IN-BROWSER AI → FOLLOW**; the HUD chip shows `PPO ONNX` once the policy is found, otherwise it reports the built-in `PD ASSIST` fallback.

The observation layout is the contract between this notebook and `PpoFollowPolicy.ts`. The browser reads the exported model's input width, so a 7-float model keeps working with range and altitude zero-padded — but if you change what the extra channels mean, retrain and re-export.
